# Big Four Indonesian Banks Stock Portfolio Analytics (2015 - 2026)
**Dataset:** Daily adjusted closing prices for BBCA, BBRI, BMRI, BBNI (Jan 2, 2015 - Jul 17, 2026).

This notebook processes the historical daily prices, performs statistical profiling (stationarity and normality testing), estimates stochastic risk metrics (Value at Risk and Expected Shortfall), analyzes rolling risk (volatility, drawdowns, and market beta), and evaluates an equal-weighted portfolio benchmark. All final analyses, interpretations, and visualizations are detailed in the project's README.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import jarque_bera, skew, kurtosis, norm
from statsmodels.tsa.stattools import adfuller

# Visualization setup
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

os.makedirs('images', exist_ok=True)
print("Setup complete. 'images' directory verified.")

### Data Loading & Integrity Check

In [ ]:
df_close = pd.read_csv('data/combined_close.csv', parse_dates=['Date'])
df_close.set_index('Date', inplace=True)

print("--- Data Dimensions ---")
print(df_close.shape)
print("\n--- First 5 Rows ---")
print(df_close.head())
print("\n--- Missing Values ---")
print(df_close.isnull().sum())

### Historical Closing Price Trends
Plots the daily adjusted closing prices for the four banks.

In [ ]:
plt.figure(figsize=(14, 7))
for col in df_close.columns:
    plt.plot(df_close.index, df_close[col], label=col, linewidth=1.5)

plt.title('Daily Adjusted Closing Prices of Big Four Indonesian Banks (2015 - 2026)', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Adjusted Close Price (IDR)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('images/price_trends.png', dpi=300, bbox_inches='tight')
plt.show()

### Log Returns Transformation
Log returns are calculated for modeling and testing: $r_t = \ln(P_t / P_{t-1})$

In [ ]:
df_log_returns = np.log(df_close / df_close.shift(1)).dropna()
print(df_log_returns.head())

### Statistical Foundations: Stationarity & Normality Testing
- **Augmented Dickey-Fuller (ADF)** is applied to returns to confirm stationarity ($H_0$: unit root present).
- **Jarque-Bera (JB)** tests returns for normality ($H_0$: normally distributed).

In [ ]:
# ADF Stationarity Test
adf_results = []
for col in df_log_returns.columns:
    stat, p, _, _, critical, _ = adfuller(df_log_returns[col])
    adf_results.append({
        'Bank': col,
        'ADF Stat': stat,
        'p-value': p,
        'Stationary (p<0.05)': 'YES' if p < 0.05 else 'NO'
    })
print("=== ADF TEST RESULTS ===")
print(pd.DataFrame(adf_results).set_index('Bank'))

# JB Normality Test
jb_results = []
for col in df_log_returns.columns:
    series = df_log_returns[col]
    jb_stat, jb_p = jarque_bera(series)
    jb_results.append({
        'Bank': col,
        'Skewness': skew(series),
        'Excess Kurtosis': kurtosis(series),
        'JB Stat': jb_stat,
        'p-value': jb_p,
        'Normal (p>0.05)': 'YES' if jb_p > 0.05 else 'NO'
    })
print("\n=== JB TEST RESULTS ===")
print(pd.DataFrame(jb_results).set_index('Bank'))

### Return Distributions vs. Theoretical Normal Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
axes = axes.flatten()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, col in enumerate(['BBCA', 'BBRI', 'BMRI', 'BBNI']):
    series = df_log_returns[col]
    sns.histplot(series, stat='density', bins=100, ax=axes[i], color=colors[i], alpha=0.5)
    
    # Fit normal distribution
    mu, std = norm.fit(series)
    xmin, xmax = axes[i].get_xlim()
    x = np.linspace(xmin, xmax, 100)
    axes[i].plot(x, norm.pdf(x, mu, std), 'k--', linewidth=2, label=f'Normal Fit (σ={std:.4f})')
    
    axes[i].set_title(f'{col} Log Return Distribution', fontweight='bold')
    axes[i].set_xlabel('Log Return')
    axes[i].set_ylabel('Density')
    axes[i].legend(fontsize=9)

plt.suptitle('Log Return Distributions vs. Theoretical Normal Curves', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('images/return_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

### Market Risk Metrics: Value at Risk (VaR) & Expected Shortfall (ES)
Computes the daily historical 95% VaR (cutoff for worst 5% returns) and 95% Expected Shortfall (average return given that loss exceeds the VaR threshold).

In [ ]:
risk_metrics = []
for col in ['BBCA', 'BBRI', 'BMRI', 'BBNI']:
    series = df_log_returns[col]
    var_95 = -np.percentile(series, 5)
    es_95 = -series[series <= -var_95].mean()
    risk_metrics.append({
        'Bank': col,
        'Daily VaR 95% (%)': var_95 * 100,
        'Daily ES 95% (%)': es_95 * 100
    })
df_risk_metrics = pd.DataFrame(risk_metrics).set_index('Bank')
print("=== DAILY RISK METRICS ===")
print(df_risk_metrics.round(4))

### Daily Log Returns vs. 95% VaR Thresholds

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
for i, col in enumerate(['BBCA', 'BBRI', 'BMRI', 'BBNI']):
    series = df_log_returns[col]
    cutoff = df_risk_metrics.loc[col, 'Daily VaR 95% (%)'] / 100
    axes[i].plot(series.index, series, color=colors[i], alpha=0.5, linewidth=0.8)
    axes[i].axhline(-cutoff, color='red', linestyle='--', linewidth=1.2, label=f'VaR 95% (-{cutoff*100:.2f}%)')
    axes[i].set_title(f'{col} Returns vs. Daily VaR 95%', fontweight='bold')
    axes[i].set_ylabel('Log Return')
    axes[i].legend(loc='lower left', fontsize=9)

plt.xlabel('Date')
plt.tight_layout()
plt.savefig('images/returns_vs_var.png', dpi=300, bbox_inches='tight')
plt.show()

### Rolling Risk Profiling: Volatility & Drawdowns
- **Annualized Rolling Volatility (21-Day)**: Spot volatility trends and clustering.
- **Maximum Drawdown (MDD)**: Historical peak-to-trough peak percentage falls.

In [ ]:
trading_days = 252
# Rolling Volatility (21-day window)
df_rolling_vol = df_log_returns[['BBCA', 'BBRI', 'BMRI', 'BBNI']].rolling(window=21).std() * np.sqrt(trading_days) * 100

plt.figure(figsize=(14, 6))
for col in df_rolling_vol.columns:
    plt.plot(df_rolling_vol.index, df_rolling_vol[col], label=col, alpha=0.8, linewidth=1.5)
plt.title('Annualized Rolling 21-Day Volatility Trends (2015 - 2026)', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Annualized Volatility (%)')
plt.legend(loc='upper right')
plt.tight_layout()
plt.savefig('images/rolling_volatility.png', dpi=300, bbox_inches='tight')
plt.show()

# Drawdowns
df_peaks = df_close[['BBCA', 'BBRI', 'BMRI', 'BBNI']].cummax()
df_drawdowns = (df_close[['BBCA', 'BBRI', 'BMRI', 'BBNI']] - df_peaks) / df_peaks

plt.figure(figsize=(14, 6))
for col in df_drawdowns.columns:
    plt.plot(df_drawdowns.index, df_drawdowns[col] * 100, label=f"{col} (Max: {df_drawdowns[col].min()*100:.2f}%)", alpha=0.75)
plt.title('Drawdown Analysis of Big Four Indonesian Banks (2015 - 2026)', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Drawdown (%)')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('images/drawdowns.png', dpi=300, bbox_inches='tight')
plt.show()

### Systemic Risk & Market Sensitivity: Rolling 60-Day Beta ($\beta$)
Estimates rolling 60-day betas for each bank relative to the Equal-Weighted Big Banks Index.

$$\beta_i = \frac{\text{Cov}(r_i, r_{index})}{\text{Var}(r_{index})}$$

In [ ]:
df_log_returns['Index'] = df_log_returns[['BBCA', 'BBRI', 'BMRI', 'BBNI']].mean(axis=1)

rolling_var = df_log_returns['Index'].rolling(window=60).var()
betas = {}
for col in ['BBCA', 'BBRI', 'BMRI', 'BBNI']:
    rolling_cov = df_log_returns[col].rolling(window=60).cov(df_log_returns['Index'])
    betas[col] = rolling_cov / rolling_var

df_betas = pd.DataFrame(betas)

plt.figure(figsize=(14, 6))
for col in df_betas.columns:
    plt.plot(df_betas.index, df_betas[col], label=col, alpha=0.8, linewidth=1.5)
plt.axhline(1.0, color='grey', linestyle='--', label='Market Beta = 1.0')
plt.title('Rolling 60-Day Beta Trends Relative to Banking Index Benchmark', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Beta')
plt.legend(loc='lower left')
plt.tight_layout()
plt.savefig('images/rolling_betas.png', dpi=300, bbox_inches='tight')
plt.show()

print("--- Average Beta Historis ---")
print(df_betas.mean().round(4))

### Portfolio Analysis: Equal-Weighted Portfolio vs. Individual Stocks
Evaluates a rebalanced Equal-Weighted (EW) Portfolio (25% allocation each) benchmarked against the four single holdings.

In [ ]:
rf_rate = 5.0
portfolio_log_return = df_log_returns[['BBCA', 'BBRI', 'BMRI', 'BBNI']].mean(axis=1)

perf_metrics = {}
for col in ['BBCA', 'BBRI', 'BMRI', 'BBNI']:
    series = df_log_returns[col]
    ann_ret = series.mean() * trading_days * 100
    ann_vol = series.std() * np.sqrt(trading_days) * 100
    sharpe = (ann_ret - rf_rate) / ann_vol
    max_dd = df_drawdowns[col].min() * 100
    perf_metrics[col] = [ann_ret, ann_vol, sharpe, max_dd]

# Portfolio Metrics
ann_ret_p = portfolio_log_return.mean() * trading_days * 100
ann_vol_p = portfolio_log_return.std() * np.sqrt(trading_days) * 100
sharpe_p = (ann_ret_p - rf_rate) / ann_vol_p

cum_port = np.exp(portfolio_log_return.cumsum())
dd_port = (cum_port - cum_port.cummax()) / cum_port.cummax()
max_dd_p = dd_port.min() * 100

perf_metrics['EW Portfolio'] = [ann_ret_p, ann_vol_p, sharpe_p, max_dd_p]

df_perf = pd.DataFrame(perf_metrics, index=['Annualized Return (%)', 'Annualized Volatility (%)', 'Sharpe Ratio', 'Max Drawdown (%)'])
print("=== PERFORMANCES COMPARISON ===")
print(df_perf.round(4))

### Cumulative Growth Simulation (Rp 1.000.000 Initial Investment)

In [ ]:
initial_capital = 1_000_000
df_cum_returns = np.exp(df_log_returns[['BBCA', 'BBRI', 'BMRI', 'BBNI']].cumsum()) * initial_capital
df_cum_returns['EW Portfolio'] = np.exp(portfolio_log_return.cumsum()) * initial_capital

# Append start date with initial value
start_date = df_close.index[0]
df_cum_returns.loc[start_date] = [initial_capital] * 5
df_cum_returns.sort_index(inplace=True)

plt.figure(figsize=(14, 7))
for col in df_cum_returns.columns:
    if col == 'EW Portfolio':
        plt.plot(df_cum_returns.index, df_cum_returns[col], label=f'{col} (Final: Rp {df_cum_returns[col].iloc[-1]:,.0f})', color='black', linewidth=2.5)
    else:
        plt.plot(df_cum_returns.index, df_cum_returns[col], label=f'{col} (Final: Rp {df_cum_returns[col].iloc[-1]:,.0f})', alpha=0.75, linewidth=1.5)

plt.title('Cumulative Return Growth of Rp 1.000.000 Investment (2015 - 2026)', fontsize=15, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Investment Value (IDR)')
plt.gca().yaxis.set_major_formatter(plt.FuncFormatter(lambda x, loc: "{:,}".format(int(x))))
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig('images/cumulative_returns.png', dpi=300, bbox_inches='tight')
plt.show()

### Return Correlations (Pearson Matrix Heatmap)

In [ ]:
corr_matrix = df_log_returns[['BBCA', 'BBRI', 'BMRI', 'BBNI']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=0.4, vmax=1.0, fmt=".3f", 
            linewidths=0.5, cbar_kws={'label': 'Pearson Correlation Coeff'})
plt.title('Log Returns Correlation Matrix Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()